# Django, Models and Data Definition

## Introduction to Django Models
---

In this lesson, you'll learn how to **define data models** in Django.

Models are Python classes that represent your database tables - they define the structure of your data.

1. [What is a Model](#What-is-a-Model),
    - [Models as Python Classes](#Models-as-Python-Classes),
    - [Model to Database Mapping](#Model-to-Database-Mapping),
2. [Field Types](#Field-Types),
    - [Text Fields](#Text-Fields),
    - [Numeric Fields](#Numeric-Fields),
    - [Date and Time Fields](#Date-and-Time-Fields),
    - [Relationship Fields](#Relationship-Fields),
3. [Field Options](#Field-Options),
    - [Common Options](#Common-Options),
    - [The choices Option](#The-choices-Option),
4. [Meta Class](#Meta-Class),
    - [Ordering and Naming](#Ordering-and-Naming),
    - [Database Table Name](#Database-Table-Name),
5. [🧠 EXERCISE 🧠, Define a Book Model](#🧠-EXERCISE-🧠,-Define-a-Book-Model).

<br>

## What is a Model

---


<div style="text-align: center;">
    <img src="../images/model.png" width="230" height="230">
</div>

### Models as Python Classes

---

A **model** is a Python class that inherits from `django.db.models.Model`.

Think of a model like a **blueprint** for your data:
- Each model maps to a **single database table**
- Each attribute of the model represents a **database column**
- Each instance of the model represents a **row** in the table

<br>

| Python Concept | Database Equivalent |
|----------------|--------------------|
| Model class | Table |
| Class attribute (field) | Column |
| Model instance | Row |
| Instance attribute | Cell value |

In [ ]:
# Example: Basic model definition (for illustration)

from django.db import models

class Blog(models.Model):
    title = models.CharField(max_length=200)   # VARCHAR(200)
    author = models.CharField(max_length=100)  # VARCHAR(100)
    published_date = models.DateField()        # DATE
    
    def __str__(self) -> str:
        return self.title

<br>

**Key points:**
- Models live in `models.py` inside your app directory
- You don't define an `id` field - Django adds it automatically as a primary key
- Field types determine both Python and database data types

<br>

### Model to Database Mapping

---

Django's **ORM (Object-Relational Mapping)** translates your Python code to SQL.

You never have to write SQL manually - Django does it for you:

<br>

**Your Python code:**
```python
class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
```

**Generated SQL (PostgreSQL):**
```sql
CREATE TABLE blog_blog (
    id SERIAL PRIMARY KEY,
    title VARCHAR(200) NOT NULL,
    author VARCHAR(100) NOT NULL,
    published_date DATE NOT NULL
);
```

<br>

**Table naming convention:** `<app_name>_<model_name>` in lowercase
- App: `blog`, Model: `Blog` → Table: `blog_blog`


<br>

## Field Types

---

Django provides many **field types** to match different kinds of data. Let's explore the most common ones.

### Text Fields

---

| Field Type | Description | Database Type |
|------------|-------------|---------------|
| `CharField` | Short text with max length | VARCHAR |
| `TextField` | Long text without limit | TEXT |
| `EmailField` | Email with validation | VARCHAR(254) |
| `URLField` | URL with validation | VARCHAR(200) |
| `SlugField` | URL-friendly text | VARCHAR(50) |

In [ ]:
# Example: Text field types

from django.db import models

class Blog(models.Model):
    # Core attributes from our current model
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()

    # Additional content and text-related fields
    content = models.TextField()
    author_email = models.EmailField()

    def __str__(self) -> str:
        return self.title

<br>

**Important:** `CharField` always requires `max_length`. This is a common source of errors!

So, if you add a new field, you need to create a new migration in most cases.

<br>

### Numeric Fields

---

| Field Type | Description | Database Type |
|------------|-------------|---------------|
| `IntegerField` | Integer (-2B to 2B) | INTEGER |
| `PositiveIntegerField` | 0 to 2B | INTEGER |
| `BigIntegerField` | Larger range | BIGINT |
| `FloatField` | Floating point | DOUBLE PRECISION |
| `DecimalField` | Exact decimals | DECIMAL |

In [ ]:
# Example: Numeric field types

from django.db import models

class Product(models.Model):
    name = models.CharField(max_length=200)
    
    # IntegerField - for whole numbers
    stock_quantity = models.IntegerField(default=0)
    
    # PositiveIntegerField - only positive values
    views_count = models.PositiveIntegerField(default=0)
    
    # DecimalField - REQUIRED: max_digits and decimal_places
    # Perfect for money - avoids floating point errors
    price = models.DecimalField(max_digits=10, decimal_places=2)
    # max_digits=10, decimal_places=2 → up to 99999999.99
    
    # FloatField - for scientific calculations where precision is less critical
    weight_kg = models.FloatField(null=True, blank=True)

In [ ]:
from django.db import models


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()

    # Single numeric attribute
    views_count = models.PositiveIntegerField(default=0)

<br>

**Best practice:** Always use `DecimalField` for money, never `FloatField`!

```python
# BAD: Floating point errors can occur
price = models.FloatField()

# GOOD: Exact decimal representation
price = models.DecimalField(max_digits=10, decimal_places=2)
```

<br>

### Date and Time Fields

---

| Field Type | Description | Database Type |
|------------|-------------|---------------|
| `DateField` | Date only | DATE |
| `DateTimeField` | Date and time | TIMESTAMP |
| `TimeField` | Time only | TIME |
| `DurationField` | Time duration | INTERVAL |

In [ ]:
# Example: Date and time fields

from django.db import models

class Event(models.Model):
    name = models.CharField(max_length=200)
    
    # DateField - just the date (no time)
    event_date = models.DateField()
    # Example: 2026-03-15
    
    # DateTimeField - date and time together
    start_datetime = models.DateTimeField()
    # Example: 2026-03-15 14:30:00
    
    # auto_now_add - automatically set when object is created
    created_at = models.DateTimeField(auto_now_add=True)
    
    # auto_now - automatically update every time object is saved
    updated_at = models.DateTimeField(auto_now=True)

In [ ]:
from django.db import models


class Blog(models.Model):
    # ...
    published_date = models.DateField()
    # ...

<br>

**Common pattern:** Most models have `created_at` and `updated_at` fields:

```python
class BaseModel(models.Model):
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)
    
    class Meta:
        abstract = True  # Won't create a table for this model
```

🔑 A `BaseModel` is essentially a reusable template for your other database models.

<br>

### Relationship Fields

---

Relationships connect models together, just like foreign keys connect tables in a database.

<br>

| Field Type | Relationship | Example |
|------------|--------------|--------|
| `ForeignKey` | Many-to-One | Many blogs → One author |
| `OneToOneField` | One-to-One | One user → One profile |
| `ManyToManyField` | Many-to-Many | Blogs ↔ Categories |

In [ ]:
# Example: ForeignKey - Many-to-One relationship (blog_blog + comments)

from django.db import models

class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    views_count = models.PositiveIntegerField(default=0)


class Comment(models.Model):
    """A comment left by a user on a specific blog post."""
    text = models.TextField()
    created_at = models.DateTimeField(auto_now_add=True)

    # ForeignKey - many comments can belong to one blog
    # on_delete=CASCADE: if the blog is deleted, all its comments are deleted too
    blog = models.ForeignKey(
        Blog, 
        on_delete=models.CASCADE, 
        related_name='comments'  # Access via: blog_instance.comments.all()
    )

    def __str__(self) -> str:
        return f"Comment on {self.blog.title}"

🔑 You don't need the create a new migration file yet.

<br>

**`on_delete` options:**

| Option | Behavior |
|--------|----------|
| `CASCADE` | Delete related objects (most common) |
| `PROTECT` | Prevent deletion if related objects exist |
| `SET_NULL` | Set to NULL (requires `null=True`) |
| `SET_DEFAULT` | Set to default value |
| `DO_NOTHING` | Do nothing (may cause errors) |

In [ ]:
# Example: ManyToManyField

from django.db import models

class Category(models.Model):
    """A category for organizing books."""
    name = models.CharField(max_length=100)


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    views_count = models.PositiveIntegerField(default=0)
    
    # ManyToManyField - books can have many categories, categories can have many books
    # Django automatically creates an intermediate table
    categories = models.ManyToManyField(
        Category,
        related_name='blog'
    )

# Usage:
# blog.categories.all()     - get all categories for a book
# category.blog.all()      - get all books in a category
# blog.categories.add(cat)  - add category to book

<br>

## Field Options

---

### Common Options

---

Every field type accepts certain **options** (keyword arguments) that configure its behavior.

<br>

| Option | Default | Description |
|--------|---------|-------------|
| `null` | `False` | If `True`, allows NULL in database |
| `blank` | `False` | If `True`, allows empty string in forms |
| `default` | - | Default value for the field |
| `unique` | `False` | If `True`, value must be unique |
| `db_index` | `False` | If `True`, creates database index |
| `help_text` | - | Help text for forms |

In [ ]:
# Example: Common field options

from django.db import models

class Book(models.Model):
    # Required field - must have a value
    title = models.CharField(max_length=200)
    
    # Optional field - can be NULL in database and empty in forms
    subtitle = models.CharField(max_length=200, null=True, blank=True)
    
    # Field with default value
    is_published = models.BooleanField(default=False)
    
    # Unique field - no two books can have the same ISBN
    isbn = models.CharField(max_length=13, unique=True)
    
    # Indexed field - faster lookups by this field
    publication_year = models.IntegerField(db_index=True)
    
    # Field with help text - displayed in admin and forms
    summary = models.TextField(
        blank=True,
        help_text="A brief description of the book (max 500 words)"
    )

<br>

**Understanding null vs blank:**

| `null` | `blank` | Meaning |
|--------|---------|--------|
| `False` | `False` | Required everywhere |
| `False` | `True` | Can be empty in forms, but stored as empty string |
| `True` | `False` | NULL in DB, but form requires value (rare) |
| `True` | `True` | Fully optional - NULL in DB, optional in forms |

<br>

**Best practice:** For text fields, prefer `blank=True` over `null=True` to avoid having two "empty" states (NULL and empty string).

<br>

### The `choices` Option

---

The `choices` option limits a field to a predefined set of values - like a dropdown in forms.

In [ ]:
from django.db import models

CATEGORY_CHOICES = [
    ('tech', 'Technology'),
    ('business', 'Business'),
    ('lifestyle', 'Lifestyle'),
    ('news', 'News'),
]


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    views_count = models.PositiveIntegerField(default=0)

    # CharField with limited choices
    category = models.CharField(
        max_length=20,
        choices=CATEGORY_CHOICES,
        default='tech'
    )

Using raw string values isn't very practical for a Django project, so it's better to use what we call Enums instead.

In [ ]:
# Example: Using IntegerChoices for cleaner code (Django 3.0+)

from django.db import models


class Category(models.IntegerChoices):
    TECH = 1, 'Technology'
    BUSINESS = 2, 'Business'
    LIFESTYLE = 3, 'Lifestyle'
    NEWS = 4, 'News'


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    views_count = models.PositiveIntegerField(default=0)

    category = models.PositiveSmallIntegerField(
        choices=Category.choices,
        default=Category.TECH
    )

# Usage:
# blog.category = Category.NEWS
# if blog.category == Category.TECH: ...

To avoid bloating `models.py` with non-model-specific constants, a better approach is to move your `Enum` definitions into a separate `enums.py` file to maintain a clean separation of concerns.

<br>

**Category enum used in this lesson:** `IntegerChoices`

```python
class Category(models.IntegerChoices):
    TECH = 1, 'Technology'
    BUSINESS = 2, 'Business'
    LIFESTYLE = 3, 'Lifestyle'
    NEWS = 4, 'News'
```

<br>

## Meta Class

---

The `Meta` inner class provides **metadata** about your model - how it should behave, be named, or be organized.

### Ordering and Naming

---

In [ ]:
from django.db import models


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    published_date = models.DateField()
    views_count = models.PositiveIntegerField(default=0)

    category = models.PositiveSmallIntegerField(
        choices=Category.choices,
        default=Category.TECH
    )
    
    class Meta:
            # The newest blogs first, then by title
            ordering = ['-published_date', 'title']
            
            # Correct names for the admin interface
            verbose_name = 'Blog post'
            verbose_name_plural = 'Blog posts'

            # Logically improved: Defining a field for the .latest() and .earliest() methods
            get_latest_by = 'published_date'

            # Performance: Indexing the field that is often sorted or filtered
            indexes = [
                models.Index(fields=['-published_date']),
                models.Index(fields=['category']),
            ]

<br>

**Common Meta options:**

| Option | Description |
|--------|-------------|
| `ordering` | Default sort order |
| `verbose_name` | Human name (singular) |
| `verbose_name_plural` | Human name (plural) |
| `db_table` | Custom database table name |
| `unique_together` | Fields that must be unique together |
| `indexes` | Database indexes to create |
| `abstract` | If `True`, don't create table for this model |

<br>

### Database Table Name

---

In [ ]:
# Example: Custom table name and constraints

from django.db import models

class BlogReview(models.Model):
    blog = models.ForeignKey('Blog', on_delete=models.CASCADE)
    rating = models.IntegerField()
    comment = models.TextField(blank=True)
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        db_table = 'blog_reviews'
        unique_together = ['blog', 'created_at']
        # Preferred modern alternative:
        # constraints = [
        #     models.UniqueConstraint(fields=['blog', 'created_at'], name='unique_blog_review')
        # ]

        # Create index for faster queries
        indexes = [
            models.Index(fields=['blog', '-created_at'])
        ]

<br>

## Complete Model Example

---

Let's put it all together with a complete, real-world model example:

In [ ]:
# Example: Complete models for the blog app
# File: blog/models.py

from django.db import models
from django.contrib.auth.models import User


class CategoryType(models.IntegerChoices):
    TECH = 1, 'Technology'
    BUSINESS = 2, 'Business'
    LIFESTYLE = 3, 'Lifestyle'
    NEWS = 4, 'News'


class Blog(models.Model):
    title = models.CharField(max_length=200)
    author = models.CharField(max_length=100)
    author_email = models.EmailField()
    slug = models.SlugField(max_length=200, unique=True)
    content = models.TextField()
    published_date = models.DateField()
    views_count = models.PositiveIntegerField(default=0)

    # IntegerChoices example used in this lesson
    category_type = models.PositiveSmallIntegerField(
        choices=CategoryType.choices,
        default=CategoryType.TECH,
    )

    class Meta:
        ordering = ['-published_date', 'title']
        indexes = [
            models.Index(fields=['slug']),
            models.Index(fields=['category_type', '-published_date']),
        ]

    def __str__(self):
        return self.title


class Comment(models.Model):
    blog = models.ForeignKey(Blog, on_delete=models.CASCADE, related_name='comments')
    content = models.TextField()
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        ordering = ['-created_at']

    def __str__(self):
        return f"Comment on {self.blog.title}"


class BlogReview(models.Model):
    blog = models.ForeignKey(Blog, on_delete=models.CASCADE, related_name='reviews')
    user = models.ForeignKey(User, on_delete=models.CASCADE, related_name='blog_reviews')
    rating = models.PositiveSmallIntegerField()
    comment = models.TextField(blank=True)
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        db_table = 'blog_reviews'
        constraints = [
            models.UniqueConstraint(fields=['blog', 'user'], name='unique_blog_review'),
            models.CheckConstraint(
                check=models.Q(rating__gte=1) & models.Q(rating__lte=5),
                name='blog_review_rating_1_to_5',
            ),
        ]
        indexes = [
            models.Index(fields=['blog', '-created_at']),
        ]

    def __str__(self):
        return f"Review {self.rating}/5 for {self.blog.title}"

#### How to remove migration (locally)

1. Check migration status:
```bash
cd mysite
python manage.py showmigrations blog # (verify if 0002 is marked with [X]).
```

2. Rollback the app to the first migration:
```bash
python manage.py migrate blog 0001
```
This triggers the reverse operation of migration 0002 (e.g., dropping a table or a column from the database).

3. Delete the second migration file:
Manually delete: `mysite/blog/migrations/0002_comment.py`.

4. Verify that models match the `0001` state:

If your `models.py` still contains changes from `0002`, Django will prompt you to create a new migration.

Run: `python manage.py makemigrations --check` (if it reports changes, you must either revert the model code or accept a new migration).

5. Optional: Local DB reset (development only):

If it's a local project and data loss is not an issue, delete db.sqlite3 and run:

`python manage.py migrate`

🔑 For consistent migration naming, I suggest using only the numeric prefix. Note that this requires some minor optimization at the beginning of the process.

1. Rename the file:
```bash
mv mysite/blog/migrations/0001_initial.py mysite/blog/migrations/0001.py
```

2. Modify the table record:
```bash
python manage.py dbshell
```
via. SQLite:
```sql
UPDATE django_migrations 
SET name = '0001' 
WHERE app = 'blog' AND name = '0001_initial';
```

3. Validate the changes:
```bash
python manage.py showmigrations blog
```

4. Prepare new migrations like:
```bash
python manage.py makemigrations blog --name 0002
```


### Mandatory field issues

🔑 Essentially, you are trying to add a mandatory (non-nullable) field to a table that already contains data. Django is asking: 'What should I put in this new column for the rows that already exist?

Is this a valid professional approach?

#### Expected result

```bash
sqlite> SELECT app, name, applied FROM django_migrations ORDER BY id;
contenttypes|0001_initial|2026-03-26 11:54:56.180012
# ...
blog|0001|2026-03-26 11:54:56.220102
sessions|0001_initial|2026-03-26 11:54:56.221115
blog|0002|2026-03-31 05:29:48.162912
```

<br>

**Notice:**
- Each model has a `__str__` method for readable representation
- Related fields use `related_name` for reverse lookups
- `IntegerChoices` keeps category values strict and explicit
- Constraints protect data quality (`UniqueConstraint`, `CheckConstraint`)

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **Model** | Python class inheriting from `models.Model` |
| **Field types** | `CharField`, `TextField`, `IntegerField`, `DecimalField`, `DateField`, `ForeignKey`, etc. |
| **Field options** | `null`, `blank`, `default`, `unique`, `choices` |
| **Meta class** | `ordering`, `verbose_name`, `db_table`, `unique_together` |
| **Best practices** | Use `DecimalField` for money, add `__str__`, use `related_name` |

<br>

### 🧠 EXERCISE 🧠, Define a Book Model

---

Create a model for a library management system:

1. Create `Library` model with:
   - `name` (CharField, max 200 chars)
   - `address` (TextField)
   - `city` (CharField, max 100 chars)
   - `established_year` (IntegerField)

2. Create `Member` model with:
   - `library` (ForeignKey to Library)
   - `first_name`, `last_name` (CharField)
   - `email` (EmailField, unique)
   - `membership_type` (CharField with choices: 'basic', 'premium', 'student')
   - `joined_date` (DateField, auto set on creation)

3. Add appropriate Meta class with:
   - Ordering by `last_name`, then `first_name`
   - `verbose_name` and `verbose_name_plural`

4. Add `__str__` methods to both models

<details>
    <summary>▶️ Solution</summary>
    
```python
# library/models.py

from django.db import models


class Library(models.Model):
    """A library with books and members."""
    name = models.CharField(max_length=200)
    address = models.TextField()
    city = models.CharField(max_length=100)
    established_year = models.IntegerField()
    
    class Meta:
        verbose_name_plural = 'libraries'
        ordering = ['name']
    
    def __str__(self):
        return f"{self.name} ({self.city})"


class Member(models.Model):
    """A library member."""
    
    class MembershipType(models.TextChoices):
        BASIC = 'basic', 'Basic'
        PREMIUM = 'premium', 'Premium'
        STUDENT = 'student', 'Student'
    
    library = models.ForeignKey(
        Library,
        on_delete=models.CASCADE,
        related_name='members'
    )
    first_name = models.CharField(max_length=100)
    last_name = models.CharField(max_length=100)
    email = models.EmailField(unique=True)
    membership_type = models.CharField(
        max_length=20,
        choices=MembershipType.choices,
        default=MembershipType.BASIC
    )
    joined_date = models.DateField(auto_now_add=True)
    
    class Meta:
        verbose_name = 'member'
        verbose_name_plural = 'members'
        ordering = ['last_name', 'first_name']
    
    def __str__(self):
        return f"{self.first_name} {self.last_name}"
```
</details>

<br>

### 🧠 EXERCISE 🧠, Field Types and Options

---

For each scenario, choose the appropriate field type and options:

1. A product **price** that needs to be exact (e.g., $19.99)
2. A user's **biography** that can be very long and is optional
3. An article **status** that can only be 'draft', 'published', or 'archived'
4. A **publication date** that should automatically record when an article is created
5. A **category** field where each product can belong to multiple categories

<details>
    <summary>▶️ Solution</summary>
    
```python
from django.db import models

# 1. Price - use DecimalField for exact money representation
price = models.DecimalField(max_digits=10, decimal_places=2)

# 2. Biography - TextField for long text, blank=True for optional
biography = models.TextField(blank=True)

# 3. Status - CharField with choices
STATUS_CHOICES = [
    ('draft', 'Draft'),
    ('published', 'Published'),
    ('archived', 'Archived'),
]
status = models.CharField(
    max_length=20,
    choices=STATUS_CHOICES,
    default='draft'
)

# 4. Publication date - DateTimeField with auto_now_add
published_at = models.DateTimeField(auto_now_add=True)

# 5. Categories - ManyToManyField
categories = models.ManyToManyField('Category', related_name='products')
```
</details>

---